# 🎵 1강. 내 노래를 숫자로 보기 (AI Pair) 🎧
**AI Human 개발자 과정 강사 김생근 · KDT 부트캠프 · 이스트소프트**

> 📌 **이 노트북의 목표**
> - 노래를 "빠르다·시끄럽다·청량하다" 같은 말이 아니라 **숫자 배열, 길이, 에너지**로 보는 첫 감각을 만듭니다.
> - librosa로 오디오를 로드하고, 파형과 RMS 에너지를 직접 계산·비교합니다.
> - 여러 곡을 같은 기준의 요약표로 비교합니다.

> **오늘의 위치**: 스포티파이의 "에너지 넘치는 곡" 플레이리스트는 사람이 하나하나 듣고 고른 게 아니라, 곡마다 계산된 숫자로 순위를 매긴 결과입니다.
> 컴퓨터는 "시끄럽다·청량하다" 같은 말을 모릅니다 — 오직 숫자 배열만 계산할 수 있습니다.
> 오늘은 그 첫 숫자, **파형과 RMS 에너지**를 직접 계산해 봅니다. 이 강은 Music AI 모듈의 출발점입니다 — Part II에서 다진 딥러닝 기초 위에, 지금부터는 "오디오"라는 새로운 입력 종류를 다룹니다. 이미지가 픽셀 격자였다면, 오디오는 시간에 따라 흔들리는 숫자 배열입니다. 이 감각이 이후 모든 강의 토대가 됩니다.

> 최신곡은 `Data/latest_music_local/`에 로컬로만 넣어 사용합니다. 원본 음원이나 잘라낸 WAV를 공유하지 않습니다.

> 🧭 **이 강이 다루지 않는 것**
> - BPM(템포)·spectral centroid 같은 정교한 피처 → **2강**에서 다룹니다.
> - 여러 곡을 자동으로 그룹 짓는 클러스터링 → **3강**에서 다룹니다.
> - 소리를 이미지(멜스펙트로그램)로 바꾸는 방법 → **4강**에서 다룹니다.
> - 지금은 딱 두 가지 — **파형**과 **RMS 에너지**만 봅니다. 욕심내지 않아도 됩니다.

## 🔑 오늘의 핵심 스캐폴딩 — 처음 만나는 함수들

| 함수/개념 | 한 줄 역할 |
|---|---|
| `librosa.load(path, sr=22050)` | 오디오 파일을 숫자 배열 `y`와 샘플링레이트 `sr`로 읽음 — 💡 "1초를 22,050번 사진 찍듯 잘게 쪼개 기록한다"는 뜻입니다 |
| `len(y) / sr` | 표본 개수 ÷ 초당 표본 수 = 곡 길이(초) |
| `sqrt(mean(y**2))` | RMS — 진폭을 제곱해 평균 낸 뒤 다시 제곱근을 씌운 "평균 크기" |
| `max(abs(y))` | peak — 가장 크게 흔들린 순간 하나 |

> **📐 오늘의 계약 5줄** — 이 다섯 줄이 맞으면 절반은 성공입니다
> 1. **입력**: WAV/MP3 파일 경로 1개
> 2. **로드 결과**: `y`(1차원 float32 배열, 대략 -1~1 범위), `sr`(정수, 이 강에서는 22,050Hz로 통일)
> 3. **길이**: `len(y) / sr` 초 — 표본 개수를 샘플링레이트로 나누면 초 단위 길이가 됩니다.
> 4. **RMS 에너지**: `sqrt(mean(y**2))` — 파형 전체의 "평균 진폭 크기"를 숫자 하나로 요약한 값입니다.
> 5. **검증 한 줄**: `0 <= peak <= 약 1.1` (mp3 디코딩 특성상 peak가 1.0을 살짝 넘기도 합니다 — 에러 아닙니다)

In [ ]:
# [패키지 확인] 처음 실행하는 환경이면 필요한 패키지를 조용히 설치합니다.
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "sklearn": "scikit-learn",
}

for module_name, package_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"[설치] {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
    else:
        print(f"[OK] {package_name}")

print("패키지 준비 완료")

> ⚠️ **경로 안내 — 본인 환경에서는 이 경로가 다를 수 있습니다**
> 아래 셀의 `find_music_root()`는 현재 폴더와 상위 폴더에서 `Data/`를 자동으로 찾습니다. 자동 탐색에 실패하면(오디오를 못 찾는다는 에러가 나면) `Data/latest_music_local/` 폴더에 직접 WAV/MP3 파일을 넣어주세요.
>
> **Colab 사용자**: 왼쪽 파일(폴더) 탭에서 `Data/latest_music_local/` 경로를 만들고 분석할 음원 파일을 업로드하면 됩니다.

In [ ]:
# [환경 설정] 오디오 분석에 필요한 도구를 준비합니다.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display

try:
    import librosa
    import librosa.display
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "librosa가 필요합니다. Colab/로컬에서 `pip install librosa soundfile` 후 다시 실행하세요."
    ) from exc

warnings.filterwarnings("ignore", category=UserWarning)

# 한글 폰트 후보 중 실제로 설치된 것만 골라 씁니다 — plt.rcParams["font.family"] 대입은
# 존재하지 않는 폰트를 넣어도 예외를 던지지 않으므로, matplotlib이 인식한 설치 폰트
# 목록(fontManager.ttflist)에서 직접 존재 여부를 확인합니다.
import matplotlib.font_manager as fm
plt.rcParams["axes.unicode_minus"] = False
_installed_fonts = {f.name for f in fm.fontManager.ttflist}
_font_candidates = ["AppleGothic", "NanumGothic", "Malgun Gothic", "DejaVu Sans"]
_selected_font = next((f for f in _font_candidates if f in _installed_fonts), _font_candidates[-1])
plt.rcParams["font.family"] = _selected_font
print(f"[폰트] 선택된 한글 폰트: {_selected_font}")

SAMPLE_RATE = 22_050
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

# 현재 폴더와 상위 폴더에서 Data/를 자동으로 찾습니다 — 노트북을 강의_AI_Pair/에서 열든
# 그 위 AI_Music/에서 열든 동작합니다. /content 경로는 Colab 표준 위치입니다.
def find_music_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd(),
        Path("/content/Music"),
        Path("/content"),
    ]
    for root in candidates:
        if (root / "Data").exists():
            return root
    return Path.cwd()


MUSIC_ROOT = find_music_root()
DATA_DIR = MUSIC_ROOT / "Data"
GENERATED_DIR = DATA_DIR / "generated_loops"
LATEST_DIR = DATA_DIR / "latest_music_local"
ATTENTION_FILE = DATA_DIR / "NewJeans-Attention.wav"

# find_music_root()가 상위 폴더를 MUSIC_ROOT로 고르는 경우를 대비한 순수 cwd 기준 폴백입니다.
LOCAL_DATA_DIR = Path.cwd() / "Data"
LOCAL_LATEST_DIR = LOCAL_DATA_DIR / "latest_music_local"

print(f"[경로] MUSIC_ROOT = {MUSIC_ROOT}")
print(f"[경로] DATA_DIR   = {DATA_DIR}")
print(f"[경로] LATEST_DIR = {LATEST_DIR}")

In [ ]:
# [데이터 찾기] 기본 샘플과 로컬 최신곡 샘플을 한 번에 모읍니다.
def collect_audio_files() -> list[Path]:
    files: list[Path] = []

    # 1순위: 사용자가 넣어둔 최신곡 로컬 샘플
    latest_dirs = [LATEST_DIR, LOCAL_LATEST_DIR]
    for latest_dir in latest_dirs:
        if latest_dir.exists():
            files.extend(sorted(p for p in latest_dir.iterdir() if p.suffix.lower() in AUDIO_EXTS))

    # 2순위: 공용 Attention 샘플
    attention_candidates = [ATTENTION_FILE, LOCAL_DATA_DIR / "NewJeans-Attention.wav"]
    for attention_path in attention_candidates:
        if attention_path.exists():
            files.append(attention_path)

    # 3순위: 수업용 합성 루프
    generated_dirs = [GENERATED_DIR, LOCAL_DATA_DIR / "generated_loops"]
    for generated_dir in generated_dirs:
        if generated_dir.exists():
            files.extend(sorted(p for p in generated_dir.glob("*.wav") if p.is_file()))

    unique: list[Path] = []
    seen = set()
    for path in files:
        key = path.resolve()
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


audio_files = collect_audio_files()
if not audio_files:
    checked = [str(LATEST_DIR), str(LOCAL_LATEST_DIR), str(GENERATED_DIR), str(LOCAL_DATA_DIR / "generated_loops")]
    raise FileNotFoundError(
        "분석할 오디오를 찾지 못했습니다. 확인한 경로:\n- " + "\n- ".join(checked)
    )

print(f"[발견] 분석 가능한 오디오 {len(audio_files)}개")
for i, path in enumerate(audio_files[:12], 1):
    print(f"  {i:02d}. {path.name}")
if len(audio_files) > 12:
    print(f"  ... 외 {len(audio_files) - 12}개")

---
## 1. 여러 샘플 먼저 들어보기

숫자를 보기 전에 먼저 귀로 비교합니다. 같은 폴더에 있는 곡 중 앞쪽 3개를 재생해보고, 어떤 곡이 더 크고 에너지가 높게 들리는지 감으로 예상해봅니다.

> 💡 **비유로 먼저 감 잡기**: 사람은 노래를 들으면 0.1초도 안 걸려 "이건 시끄럽다"를 판단합니다. 하지만 컴퓨터에게 "시끄럽다"는 저절로 아는 개념이 아닙니다 — 파형(공기 떨림의 기록)과 RMS(그 떨림의 평균 크기)라는 두 숫자로 "시끄럽다"를 정의해줘야 합니다. 오늘 하는 일은 사람의 직관 하나를 컴퓨터가 계산 가능한 숫자로 번역하는 첫 실습입니다.

In [ ]:
# [Step 1] 대표 샘플 여러 개를 먼저 들어봅니다.
preview_files = audio_files[: min(3, len(audio_files))]

for idx, path in enumerate(preview_files, 1):
    print(f"[{idx}] {path.name}")
    display(Audio(filename=str(path)))

# 이후 파형/RMS 상세 분석에는 첫 번째 샘플을 사용합니다.
sample_path = preview_files[0]
print()
print(f"[상세 분석 선택] {sample_path.name}")


> ▶ **실행 전 예측** — 아래 셀들을 실행하면 무슨 일이 벌어질지 먼저 적어보세요.
> - 방금 들은 3곡 중 어느 곡의 RMS 에너지가 가장 높을 것 같나요? 직접 들었을 때의 느낌을 적어두세요.
> - 파형 그래프는 노래 전체에서 어떤 모양일 것 같나요 — 처음부터 끝까지 비슷한 크기일까요, 구간마다 오르내릴까요?
> - 나중에 "결과 해석"에서 이 예측과 실제 계산값을 비교합니다.

In [ ]:
# [Step 2] librosa는 오디오를 숫자 배열 y와 샘플링레이트 sr로 읽습니다.
y, sr = librosa.load(sample_path, sr=SAMPLE_RATE, mono=True)
duration = librosa.get_duration(y=y, sr=sr)
rms_value = float(np.sqrt(np.mean(y ** 2)))
peak_value = float(np.max(np.abs(y)))

summary = pd.DataFrame([
    {
        "file": sample_path.name,
        "samples": len(y),
        "sample_rate": sr,
        "duration_sec": round(duration, 2),
        "rms_energy": round(rms_value, 4),
        "peak": round(peak_value, 4),
    }
])
summary

### 📐 코드를 읽기 전에 — 방금 나온 숫자들의 정체

| 코드 | 하는 일 | 비유 |
|---|---|---|
| `librosa.load(path, sr=22050, mono=True)` | 파일을 초당 22,050개 숫자로 바꿔 읽음 | 1초를 22,050컷의 스틸사진으로 잘게 쪼개 기록 |
| `len(y) / sr` (duration) | 표본 개수 ÷ 초당 표본 수 = 초 | 총 사진 수 ÷ 초당 컷 수 = 영상 길이 |
| `sqrt(mean(y**2))` (RMS) | 진폭을 제곱(부호 제거) → 평균 → 제곱근 | 소리의 "평균 크기"를 하나의 대표값으로 요약 |
| `max(abs(y))` (peak) | 가장 크게 흔들린 순간 하나만 포착 | RMS가 "평균 볼륨"이라면 peak는 "가장 시끄러웠던 찰나" |

RMS 계산에서 제곱을 하는 이유: 파형은 양수·음수를 오가며 진동하므로 그냥 평균을 내면 서로 상쇄되어 거의 0에 가까워집니다. 제곱하면 모두 양수가 되어 "크기"만 남고, 마지막에 제곱근으로 원래 단위로 되돌립니다.

In [ ]:
# [Step 3] 파형은 시간에 따라 공기가 얼마나 흔들렸는지 보여줍니다.
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(12, 3.5))
librosa.display.waveshow(y, sr=sr, ax=ax, color="#2563eb")
ax.set_title(f"Waveform — {sample_path.name}")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("Amplitude")
# librosa.display.waveshow는 기본적으로 mm:ss 형식으로 축을 그립니다 —
# 아래 Step 4 그래프(순수 초 단위)와 눈금 표기를 통일합니다.
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.0f}"))
ax.grid(alpha=0.25)
plt.show()

In [ ]:
# [Step 4] 짧은 구간별 RMS를 보면 에너지가 커지는 순간이 보입니다.
# 💡 일상 비유: frame_length=2048(약 0.09초 분량)만큼 셔터를 열어 한 번에 관찰하고,
#    hop_length=512(약 0.02초)씩 자리를 옮기며 겹쳐 찍습니다 — 겹쳐 찍으므로 급격한 에너지 변화도 놓치지 않습니다.
frame_rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
times = librosa.frames_to_time(np.arange(len(frame_rms)), sr=sr, hop_length=512)

# 원본은 프레임 단위(약 0.02초)라 168초 전체를 그리면 촘촘한 노이즈로 보입니다.
# 2초 이동평균을 함께 그려 "큰 흐름(어디가 상대적으로 더 큰 에너지인가)"이 보이게 합니다.
smooth_window = max(1, int(2.0 * sr / 512))  # 약 2초 분량 프레임 수
kernel = np.ones(smooth_window) / smooth_window
frame_rms_smooth = np.convolve(frame_rms, kernel, mode="same")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(times, frame_rms, color="#dc2626", linewidth=0.6, alpha=0.35, label="원본(프레임 단위)")
ax.plot(times, frame_rms_smooth, color="#991b1b", linewidth=2.2, label="2초 이동평균(흐름 파악용)")
ax.fill_between(times, frame_rms_smooth, alpha=0.15, color="#dc2626")
ax.set_title(f"RMS Energy — {sample_path.name}")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("RMS")
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.25)
plt.show()

### 📊 그래프 읽는 법 — 방금 그린 두 그래프 뜯어보기

**Step 3 파형(Waveform)**
- 가로축 = 시간(초), 세로축 = 진폭(amplitude, 대략 -1~1 범위 — 1에 가까울수록 공기가 크게 흔들린 순간)
- 이 예시곡처럼 **거의 전 구간이 ±1.0 가까이 꽉 차 보이는 경우도 있습니다** — 현대 팝 음악은 마스터링(loudness 압축)으로 조용한 구간 없이 평균 음량 자체를 끝까지 높게 유지하는 경우가 많기 때문입니다. 이건 오디오가 잘못된 게 아니라 "이 곡의 제작 방식"을 보여주는 것 — 클래식·어쿠스틱 곡을 넣으면 조용한 인트로·다이내믹이 뚜렷하게 보입니다(직접 다른 파일로 바꿔 실행해 비교해보세요).

**Step 4 RMS 에너지**
- 가로축 = 시간(초), 세로축 = 그 구간의 평균 에너지 크기(RMS)
- 연한 선(원본)은 0.02초 단위라 168초 전체를 보면 촘촘한 노이즈처럼 보입니다 — 그래서 **진한 선(2초 이동평균)** 을 함께 그려 큰 흐름만 골라냈습니다. 진한 선을 기준으로 보세요.
- 대략적인 기준선(정답이 아니라 감을 잡기 위한 눈대중 기준입니다):
  - RMS 0.1 이하 → 상대적으로 조용한 구간
  - RMS 0.1~0.25 → 보통 밀도의 구간
  - RMS 0.25 이상 → 에너지가 높은 구간(다만 이 곡처럼 평균 자체가 0.3 안팎으로 높은 마스터링 곡은 곡 전체가 이 범위에 몰릴 수 있습니다)
  - 이 노트북에 여러분 곡을 넣어보면 곡마다 이 기준선의 의미가 다르게 나타날 수 있습니다 — 절대 기준이 아니라 "이 곡 안에서 상대적으로 어디가 더/덜 에너지가 있는가"를 보는 용도입니다.

### 결과 해석 — "실행 전 예측"과 비교해보기

이 노트북을 실제로 실행하면 `sample_path`는 `latest_music_local/` 폴더에서 이름 순으로 가장 앞에 오는 곡( Your Shampoo Scent In The Flowers.mp3)이 선택되고, 다음과 같은 실측값이 나옵니다(seed 없이도 같은 파일·같은 SAMPLE_RATE라 재현되는 값입니다):

- **samples**: 3,717,344개, **sample_rate**: 22,050Hz → **duration**: 168.59초
- **RMS 에너지**: 0.3056, **peak**: 1.0761 (mp3 디코딩 특성상 peak가 1.0을 살짝 넘습니다 — 정상입니다)

파형이 크게 흔들리는 구간은 소리가 크거나 악기가 많이 겹치는 순간입니다. RMS는 "평균적으로 얼마나 에너지가 있는가"를 보여주며, 같은 노래라도 벌스·후렴·브릿지에서는 파형과 RMS 모양이 달라집니다. 다만 이 곡처럼 마스터링으로 전체 음량이 이미 높게 압축된 트랙은 파형·RMS만으로 벌스/후렴 경계가 뚜렷하게 안 보일 수 있습니다 — Step 4의 진한(이동평균) 곡선에서 그나마 상대적으로 높은/낮은 구간을 비교하는 정도로 활용하세요. 다이내믹 레인지가 큰 곡(어쿠스틱·클래식 등)을 넣으면 이 경계가 훨씬 뚜렷하게 보입니다.

> 📌 **실측값은 강사 환경 기준 예시값입니다**
> 위 samples/duration/RMS/peak 숫자는 강사가 실행했을 때 폴더에서 선택된 특정 파일 기준 결과입니다. 여러분이 직접 실행하면 곡 구성·환경에 따라 다른 숫자가 나옵니다 — 확인할 것은 값 자체가 아니라 **값들 사이의 관계**(어떤 곡이 더 크게/작게 들리는가, 파형이 어디서 오르내리는가)입니다.

In [ ]:
# [Step 5] 여러 샘플을 같은 기준으로 요약합니다.
def summarize_audio(path: Path) -> dict:
    y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=30)
    return {
        "file": path.name,
        "path": str(path),
        "duration_sec": round(librosa.get_duration(y=y, sr=sr), 2),
        "sample_rate": sr,
        "rms_energy": round(float(np.sqrt(np.mean(y ** 2))), 4),
        "peak": round(float(np.max(np.abs(y))), 4),
    }

summary_df = pd.DataFrame([summarize_audio(path) for path in audio_files[:10]])
rms_rank_df = summary_df.sort_values("rms_energy", ascending=False).reset_index(drop=True)
rms_rank_df[["file", "duration_sec", "sample_rate", "rms_energy", "peak"]]

### 🤔 반직관적 결과 해설 — 합성 루프가 왜 RMS 3위인가

표를 보면 `ambient_70bpm_soft_15s.wav`(RMS 0.1985)가 실제 K-pop 곡인 `사랑하게 될 거야`(0.1397)·`예뻤어 You Were Beautiful`(0.1236)·`Grown Ups`(0.0964)보다 RMS가 더 높습니다. 잔잔하게 들리는 ambient 트랙이 아이돌 곡보다 '평균 에너지'가 높다는 건 직관과 어긋나 보일 수 있습니다.

또 하나 눈에 띄는 패턴 — 합성 루프 5곡(ambient·ballad·dance·funk·citypop)의 **peak가 전부 0.8799로 완전히 동일**합니다.

두 현상 모두 같은 원인에서 나옵니다: 합성 루프는 생성 시 진폭이 일정 수준으로 **정규화(normalize)**된 신호입니다. 정규화된 신호는 조용한 구간·무음 구간이 거의 없어 파형 전체가 고르게 채워지므로, 인트로·브릿지처럼 조용한 구간이 섞인 실제 녹음곡보다 오히려 평균 에너지(RMS)가 높게 나올 수 있습니다. peak가 5곡 모두 0.8799로 똑같은 것도 같은 정규화 파이프라인이 동일한 최대 진폭 기준을 적용했기 때문입니다.

> → 실무 결론: **RMS·peak 같은 신호 통계는 '사람이 듣기에 시끄럽다'와 항상 일치하지 않습니다.** 정규화된 합성 데이터를 실제 녹음곡과 같은 잣대로 비교할 때는 이 차이를 먼저 감안해야 합니다.

### 🔍 확인해보기 — 상위 랭킹을 직접 들어보며 검증

방금 표에서 RMS 순위를 숫자로만 봤습니다. 이번엔 순위를 실제로 들어보며 "숫자가 귀로 느낀 것과 일치하는가"를 검증합니다. 실제 실행 결과 상위 3곡은 `NewJeans-Attention.wav`(RMS 0.2619) → ` Your Shampoo Scent In The Flowers.mp3`(RMS 0.2509) → `ambient_70bpm_soft_15s.wav`(RMS 0.1985) 순이었습니다.

> 🎲 **불확실성 참고**: Step 2의 RMS(0.3056)와 Step 5 요약표의 같은 곡 RMS(0.2509)가 서로 다른 걸 눈치채셨나요? 이유는 계산 구간이 다르기 때문입니다 — Step 2는 전체 168초를 다 썼고, Step 5는 `summarize_audio`가 앞부분 30초만 잘라 씁니다(`duration=30`). **같은 곡이라도 "어느 구간을 측정했는가"에 따라 RMS 숫자가 달라집니다** — RMS는 곡의 절대적 속성이 아니라 "측정 창(window)에 따라 달라지는 값"이라는 것을 기억해 두세요.

> 📌 위 순위·RMS 수치도 강사 환경에서 나온 예시값입니다. 여러분의 `audio_files` 구성에 따라 순위가 달라질 수 있습니다 — 중요한 것은 절대 수치가 아니라 '합성 루프가 K-pop보다 RMS가 높게 나올 수도 있다'는 관계 자체입니다(바로 위 해설 참고).

In [ ]:
# [Step 6] RMS가 높은 샘플을 직접 들어보며 숫자와 감상을 비교합니다.
top_n = min(3, len(rms_rank_df))
print(f"RMS 에너지 상위 {top_n}개 샘플을 재생합니다.")

for idx, row in rms_rank_df.head(top_n).iterrows():
    print(f"[{idx + 1}] RMS={row['rms_energy']:.4f} | peak={row['peak']:.4f} | {row['file']}")
    display(Audio(filename=row["path"]))

print("질문: RMS가 가장 높은 곡이 실제로도 가장 크게 느껴졌나요?")

### 비교 포인트

- RMS가 높으면 평균 에너지가 크다는 뜻이지만, 반드시 "가장 좋게 들린다"는 뜻은 아닙니다.
- peak가 높아도 RMS가 낮으면 순간적으로만 큰 소리일 수 있습니다.
- 실제 청감은 볼륨, 악기 배치, 저음/고음 비율, 압축 방식의 영향을 함께 받습니다.

### 마무리 질문

가장 RMS가 높은 곡은 실제로도 가장 시끄럽게 들렸나요? 아니라면 이유가 무엇일까요?

> 🤖 **AI Agent에서 이렇게 씁니다**: 오늘 계산한 RMS·파형은 숫자 몇 개처럼 보이지만, 이 Music AI 모듈의 뒷부분 전체가 이 숫자 위에 서 있습니다. 5강~8강에서는 이런 오디오 피처를 RandomForest·ResNet-18 같은 분류기의 입력으로 넘겨 "이 곡은 어떤 장르인가"를 판단하고, Phase2~3의 LangGraph 에이전트는 "에너지 높은 K-pop 찾아줘" 같은 사용자 요청을 처리할 때 결국 지금 배운 RMS 같은 정량적 근거를 도구(tool) 호출 결과로 받아 판단합니다. 에이전트가 "그럴듯하게 말하는 것"과 "근거 있는 숫자로 판단하는 것"의 차이가 바로 오늘 만든 이 계산에서 시작됩니다.

### 🔎 잠깐 하나만 더 — zero-crossing rate 먼저 살짝 보기

RMS가 "파형이 얼마나 크게 흔들렸는가(에너지)"를 재는 숫자였다면, **zero-crossing rate(영점 교차율)**는 "파형이 얼마나 자주 부호를 바꿨는가(거칠기)"를 재는 숫자입니다. 파형은 0을 기준으로 위·아래를 오가는데, 그 부호가 바뀌는 횟수를 프레임 단위로 세면 됩니다.

> 💡 비유로 먼저 감 잡기: 첼로·베이스 같은 낮은음은 한 번 출렁이는 데 시간이 오래 걸려 부호가 천천히 바뀝니다. 반대로 하이햇·박수처럼 거칠고 노이즈에 가까운 소리는 아주 짧은 시간에도 부호가 수없이 바뀝니다. 즉 zero-crossing rate가 높을수록 "거칠거나 노이즈에 가까운" 소리일 가능성이 큽니다.

`librosa.feature.zero_crossing_rate(y)`는 Step 4에서 RMS를 프레임 단위로 구했던 것과 같은 방식으로 동작합니다 — y를 짧은 구간(frame)으로 나눠 각 구간의 zero-crossing rate를 배열로 반환합니다. 이 배열을 평균 내면 신호 전체를 대표하는 숫자 하나가 됩니다.

아래 셀은 실제 곡이 아니라 **순수하게 만든 저음·고음·노이즈 신호 3개**로 이 API 사용법과 "부호가 자주 바뀔수록 값이 커진다"는 감각만 먼저 확인합니다 — 실제 audio_files에 이 피처를 적용해보는 건 바로 아래 Solo 레벨 2에서 직접 해봅니다. (BPM·spectral centroid 같은 나머지 피처는 2강에서 본격적으로 다룹니다.)

In [ ]:
# [Step 7] zero-crossing rate API를 먼저 작은 예시 신호로 확인합니다 (실제 오디오 적용은 Solo 레벨 2에서 직접)
sr_demo = 22050
t = np.linspace(0, 1, sr_demo, endpoint=False)

low_tone = np.sin(2 * np.pi * 5 * t)                                  # 1초에 5번 진동 — 낮은음 예시
high_tone = np.sin(2 * np.pi * 50 * t)                                # 1초에 50번 진동 — 높은음 예시
noise = np.random.default_rng(42).uniform(-1, 1, sr_demo)             # 완전 무작위 신호 — 타악기·노이즈 예시

for name, wave in [("저음 예시(5Hz)", low_tone), ("고음 예시(50Hz)", high_tone), ("노이즈", noise)]:
    zcr = librosa.feature.zero_crossing_rate(wave.astype(np.float32))[0].mean()
    print(f"{name}: zero-crossing rate 평균 = {zcr:.4f}")

---
## 🤝 AI Pair 섹션 — 오디오를 숫자로 읽기

> **목표**: AI를 답 베끼는 도구가 아니라 *내 코드·판단을 검증해주는 동료*로 쓴다. (읽기용 — 실습 시간엔 핵심만 따라가도 됩니다)

| 단계 | 내가 하는 것 | AI가 하는 것 |
|---|---|---|
| 1️⃣ Solo | 먼저 직접 작성/판단 | (아직 X) |
| 2️⃣ Review | 내 코드·판단을 제출 | 리뷰 + 근거 설명 |
| 3️⃣ Debug | AI가 준 "조용히 틀린" 코드의 결함 찾기 | 의도적 버그 제공 |
| 4️⃣ Prompt Card | 실험 설계 프롬프트 익히기 | — |

> 🎨 **난이도 표시**: 🟢초보 = 지금까지 배운 함수를 그대로 재사용 · 🟡공통 = 새 함수 하나를 추가로 찾아 적용 · 🔴개발자 = 로직/함수를 직접 설계

### 1️⃣ Solo — 먼저 스스로 풀어보세요
💡 *AI에게 물으면 30초 만에 답이 나오지만, 지금 막히는 지점이 이해가 가장 깊어지는 순간입니다.*

### ✏️ [Solo 레벨 1 · 🟢초보] 직접 작성해 보세요 — 아직 안 들어본 나머지 오디오 파일의 RMS·peak 계산하기

힌트: audio_files[3:] (앞 3개는 이미 들었으니 그 다음부터)에서 파일 하나를 골라
summarize_audio() 함수를 재사용해 결과를 출력하세요.

In [ ]:
my_path = ...  # TODO: audio_files[3:]에서 파일 하나를 선택하세요
result = ...   # TODO: summarize_audio(my_path)를 호출하세요
result

### ✏️ [Solo 레벨 2 · 🟡공통] 직접 작성해 보세요 — zero_crossing_rate 추가로 계산하기

힌트: librosa.feature.zero_crossing_rate(y)[0].mean() 로 평균 zero-crossing rate를 구할 수 있습니다.
sample_path 오디오에 대해 이 값을 계산해 출력하세요.
(바로 위 Step 7에서 API 사용법과 '값이 크면 거칠다'는 감각을 이미 확인했습니다 — 이제 실제 y에 적용해보세요.
BPM·spectral centroid 등 나머지 피처는 2강에서 본격적으로 다룹니다.)

In [ ]:
zcr_of_sample = ...  # TODO: 전역 변수 y로 zero-crossing rate 평균을 계산하세요
zcr_of_sample

### ✏️ [Solo 레벨 3 · 🔴개발자] 직접 작성해 보세요 — 전체 audio_files 순회 + RMS 상·하위 비교

힌트:
1) audio_files 전체(10개 제한 없이)에 summarize_audio()를 적용해 DataFrame을 만드세요.
2) RMS 기준 상위 3곡과 하위 3곡을 각각 출력하세요.
3) 상위 3곡과 하위 3곡의 파일 종류(생성 루프 vs 실제 곡)에 어떤 경향이 있는지 한 줄로 적어보세요.

In [ ]:
def compare_top_bottom_rms():
    # 1) audio_files 전체를 순회하며 summarize_audio()로 요약 DataFrame 만들기 (Step 5와 같은 패턴, 10개 제한만 없이)
    # full_df = ...  # TODO: pd.DataFrame([summarize_audio(p) for p in audio_files])

    # 2) RMS 기준 상위 3곡 / 하위 3곡 뽑기 (Step 5의 sort_values를 재사용)
    # top3 = ...     # TODO: full_df.sort_values("rms_energy", ascending=False).head(3)
    # bottom3 = ...  # TODO: full_df.sort_values("rms_energy", ascending=True).head(3)

    # 3) 상위/하위 파일 종류(생성 루프 vs 실제 곡) 경향을 print로 한 줄 남기기
    # TODO: 여기에 작성하세요
    pass

compare_top_bottom_rms()

### 2️⃣ Review
아래 프롬프트를 복사해 ChatGPT/Claude에 붙여넣으세요.
```text
librosa로 오디오를 로드해 RMS 에너지(sqrt(mean(y**2)))와 peak(max(abs(y)))를 계산했습니다.
같은 곡이라도 전체 길이(168초)로 계산한 RMS(0.3056)와 앞 30초만으로 계산한 RMS(0.2509)가 다르게 나왔습니다.
1) 왜 측정 구간(window)에 따라 RMS가 달라지는지, 2) 곡 전체 RMS와 30초 RMS 중 어느 쪽이 "이 곡의 평균 에너지"를 더 대표한다고 볼 수 있는지,
3) 만약 여러 곡을 공정하게 비교하려면 측정 구간을 어떻게 통일해야 하는지 각각 근거를 들어 설명해줘.
```

> ⏸️ **선택 실행 (Optional)**: 아래 코드형 Review 셀은 로컬 LLM 서버(LM Studio/Ollama) 또는 클라우드 API 키가 필요합니다 — 1강 필수 실행이 아닙니다.
> 서버가 없으면 `Connection error`가 출력되는데, 이는 **여러분 환경의 문제가 아니라 예상된 정상 동작**입니다. 바로 위 2️⃣ Review의 텍스트 프롬프트를 ChatGPT/Claude 웹에 직접 붙여넣는 것만으로도 충분합니다.

In [ ]:
# ── 2️⃣ Review (코드형, Optional) — 로컬/클라우드 LLM에게 내 코드·판단을 리뷰받기 ──
# [사전조건] 로컬: LM Studio(00-1)/Ollama(00-2) 서버 실행  |  클라우드: OPENROUTER/OPENAI 키 설정
# openai 패키지가 없으면 아래 주석을 풀어 한 번만 실행하세요 (이미 있으면 건너뛰기)
# !pip install openai
import os
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

if OpenAI is None:
    print("[안내] openai 패키지가 없어 이 Review 셀을 건너뜁니다 — `!pip install openai` 후 다시 실행하세요(선택 사항입니다).")
else:
    PROVIDER = "lmstudio"   # "lmstudio" | "ollama" | "openrouter" | "openai"  ← 한 줄만 바꾸면 전환
    PROVIDERS = {
        "lmstudio":   {"base_url": "http://localhost:1234/v1",  "model": "local-model",          "api_key": "lm-studio"},
        "ollama":     {"base_url": "http://localhost:11434/v1", "model": "llama3.2",             "api_key": "ollama"},
        "openrouter": {"base_url": "https://openrouter.ai/api/v1", "model": "anthropic/claude-3.5-sonnet", "api_key": os.getenv("OPENROUTER_API_KEY", "")},
        "openai":     {"base_url": "https://api.openai.com/v1", "model": "gpt-4o-mini",          "api_key": os.getenv("OPENAI_API_KEY", "")},
    }
    cfg = PROVIDERS[PROVIDER]
    client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])

    review_prompt = f"""librosa로 오디오를 로드해 RMS 에너지(sqrt(mean(y**2)))와 peak(max(abs(y)))를 계산했습니다.
    같은 곡({sample_path.name})을 전체 길이({duration:.1f}초)로 계산한 RMS({rms_value:.4f})와
    Step 5에서 앞 30초만으로 계산한 RMS(약 0.25)가 다르게 나왔습니다.
    1) 왜 측정 구간(window)에 따라 RMS가 달라지는지,
    2) 여러 곡을 공정하게 비교하려면 측정 구간을 어떻게 통일해야 하는지 설명해줘."""

    try:
        resp = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": review_prompt}],
            temperature=0.3,
        )
        print(resp.choices[0].message.content)
    except Exception as e:
        print(f"[안내] {PROVIDER} 서버에 연결하지 못했습니다 — 로컬 LLM 서버를 켜거나 PROVIDER를 바꿔보세요.")
        print(f"오류: {e}")

### 3️⃣ Debug — "조용히 틀린" 코드 찾기
아래는 **실행은 되고 에러도 안 나지만, RMS를 잘못 계산하는** 코드입니다. 결함을 찾아 고치세요.

| 항목 | 내용 |
|---|---|
| 증상 | 에러 메시지 없이 실행되지만, 출력된 값이 RMS 정의(제곱 → 평균 → 제곱근)와 다른 값입니다. |
| 힌트 | RMS는 반드시 "제곱 → 평균 → 제곱근" 세 단계를 거쳐야 합니다. 아래 코드가 이 세 단계를 모두 거치는지 한 줄씩 확인하세요. |

먼저 아래 코드 셀만 실행해 버그 버전의 결괏값을 확인하고, 무엇이 잘못됐는지 스스로 짚어본 뒤 그다음 셀에서 정답과 비교하세요.

In [ ]:
# [Debug] 아래 코드는 에러 없이 실행됩니다 — 그런데 RMS가 아닌 다른 값을 냅니다. 결함을 찾아 고쳐보세요.
def compute_rms_bug(y):
    # ⚠️ 여기를 의심해보세요 — 정의대로라면 "제곱 → 평균 → 제곱근"이어야 합니다.
    return float(np.mean(np.abs(y)))

rms_bug = compute_rms_bug(y)
print(f"버그 버전(mean of abs): {rms_bug:.4f}")

🔽 **정답과 비교하기** — 위에서 나온 값이 왜 틀렸는지 스스로 답을 적어본 뒤 아래 셀을 실행해 정답과 비교하세요.

In [ ]:
# [Debug 정답] RMS 정의(제곱 → 평균 → 제곱근)를 그대로 따른 버전입니다.
rms_correct = float(np.sqrt(np.mean(y ** 2)))
print(f"버그 버전(mean of abs): {rms_bug:.4f}")
print(f"정상 버전(RMS)        : {rms_correct:.4f}")

💭 **생각해 볼 점**:
- 버그 버전은 절댓값의 평균(MAD, Mean Absolute Deviation)을 계산합니다 — RMS와 "파형의 크기를 요약한다"는 목적은 같아 보이지만 수학적으로 다른 값입니다. 왜 둘이 다른 숫자가 나올까요? (힌트: 제곱은 큰 값에 더 큰 가중치를 줍니다 — 순간적으로 크게 튄 구간이 있으면 RMS가 MAD보다 더 크게 반응합니다.)
- 실제 실행 결과, 같은 파형에서 RMS는 0.3056, MAD(버그 버전)는 0.2332로 RMS가 항상 더 크게 나옵니다. 이 부등식(RMS ≥ MAD)이 항상 성립하는 이유를 수학적으로 설명할 수 있나요?
- 이 버그가 위험한 이유: 에러가 안 나고, 숫자도 "그럴듯하게" 작은 값이 나와서 눈으로 봤을 때 잘못됐다는 걸 알아채기 어렵습니다 — 왜 원래 정의(제곱-평균-제곱근)를 정확히 지켜야 하는지 보여주는 사례입니다.

### 4️⃣ Prompt Card
📝 **카드 1~2** (이 단원 실험 설계형):
```text
1. "RMS 계산 구간을 30초 대신 5초/60초로 바꾸면 순위(rms_rank_df)가 어떻게 달라질지 먼저 예측한 뒤, AI에게 근거를 묻고 실행해서 확인해줘."
2. "peak가 1.0을 넘는 이유(mp3 디코딩 특성)를 AI에게 물어보고, WAV 파일에서도 peak가 1.0을 넘을 수 있는지 같이 확인해줘."
```
🎯 **마무리 체크**: [ ] Solo 직접 [ ] Review 실행검증 [ ] Debug 결함 찾음 [ ] Prompt Card 1개 내 노트에 기록

---
## 📚 [세션 요약] 내 노래를 숫자로 보기

> 🎯 **핵심 Takeaways**
> 1. **이론적 근거**: RMS(제곱-평균-제곱근)는 진동하는 파형을 "평균 크기" 숫자 하나로 요약하는 표준적인 방법입니다.
> 2. **실무적 활용**: 같은 곡이라도 측정 구간(전체 vs 30초)에 따라 RMS 값이 달라진다 — 여러 곡을 비교할 때는 반드시 같은 기준(같은 길이)으로 재야 공정한 비교가 됩니다.
>
> ➡️ **다음 단계**: `2강_요즘음악_에너지템포비교(AI_Pair).ipynb` — BPM(템포), spectral centroid(음색의 밝기), zero crossing rate 같은 피처로 요즘 음악을 더 정교하게 비교합니다.